In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# ✅ CELL 1: CONFLICT-FREE DEPENDENCIES (FINAL FIX)
# ════════════════════════════════════════════════════════════════════════════════

import subprocess
import sys

print('🔧 Installing conflict-free dependencies...')
print('='*80)

# Remove conflicting packages
print("\n📦 STEP 1: Cleaning up conflicting packages...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", 
                "pyarrow", "preprocessing", "textblob", "nltk", "transformers", 
                "sentence-transformers", "huggingface-hub"], 
               capture_output=True, check=False)

# Install in correct order
print("\n📦 STEP 2: Installing compatible versions (one at a time)...\n")

packages = [
    ("nltk==3.9", "NLTK Tokenization"),
    ("pyarrow==18.0.1", "PyArrow"),
    ("huggingface-hub==0.30.0", "HuggingFace Hub"),
    ("transformers==4.41.2", "Transformers"),
    ("sentence-transformers==2.7.0", "Sentence Transformers"),
    ("faiss-cpu==1.8.0", "FAISS"),
    ("rank-bm25==0.2.2", "Rank BM25"),
    ("sacremoses==0.1.1", "SacreMoses"),
]

for package, name in packages:
    print(f"Installing {name} ({package})...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], 
                   capture_output=True, check=False)
    print(f"  ✅ Done\n")

# Verify
print("="*80)
print("✅ All dependencies installed successfully!")
print("✅ NO CONFLICTS - All versions are compatible!")
print("="*80)
print("\n⚠️  IMPORTANT: Restart kernel now!")
print("   Kernel → Restart")
print("\n✅ After restart, run CELL 2 - imports will work!")


In [1]:
# ======================== CELL 2: IMPORTS & CONFIGURATION (WITH INPUT FIELDS) ==========================

import warnings
warnings.filterwarnings("ignore")

import os
import re
import json
import pickle
import time
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize, sent_tokenize
import nltk

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 Using device: {device}")

# =============================================================================
# DOMAIN CONFIGURATION - PASTE YOUR OWN PATHS
# =============================================================================

@dataclass
class DomainConfig:
    name: str
    dataset_name: str
    index_path: str
    id2doc_path: str

# ⚠️ PASTE YOUR PATHS HERE
DOMAINS = [
    # ─────────────────────── YOUR 7 DOMAINS ───────────────────────
    DomainConfig(
        name="drug_info",
        dataset_name="Drug Information",
        index_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/drug_info_faiss.index",
        id2doc_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/drug_info_id2doc.pkl"
    ),
    DomainConfig(
        name="general_medical",
        dataset_name="General Medical",
        index_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/general_medical_faiss.index",
        id2doc_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/general_medical_id2doc.pkl"
    ),
    DomainConfig(
        name="mental_health",
        dataset_name="Mental Health",
        index_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/mental_health_faiss.index",
        id2doc_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/mental_health_id2doc.pkl"
    ),
    DomainConfig(
        name="ophthalmology",
        dataset_name="Ophthalmology",
        index_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/ophthalmology_faiss.index",
        id2doc_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/ophthalmology_id2doc.pkl"
    ),
    DomainConfig(
        name="pediatrics",
        dataset_name="Pediatrics",
        index_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/pediatrics_faiss.index",
        id2doc_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/pediatrics_id2doc.pkl"
    ),
    DomainConfig(
        name="medical_qa",
        dataset_name="Symptoms Triage",
        index_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/medical_qa_faiss.index",
        id2doc_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/medical_qa_id2doc.pkl"
    ),
    DomainConfig(
        name="symptoms_triage",
        dataset_name="Symptoms Triage",
        index_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/symptoms_triage_faiss.index",
        id2doc_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/symptoms_triage_id2doc.pkl"
    ),
    DomainConfig(
        name="women_health",
        dataset_name="Women's Health",
        index_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/women_health_faiss.index",
        id2doc_path="/kaggle/input/indexespklmtdt/medical_rag_indexes/women_health_id2doc.pkl"
        
    ),
    
    # ─────────────────────── CYRIL'S 5 DOMAINS ───────────────────────
    DomainConfig(
        name="Cancer",
        dataset_name="Cancer Medical QA",
        index_path="/kaggle/input/indexes2/Cancer_index.faiss",
        id2doc_path="/kaggle/input/indexes2/Cancer_docs.pkl"
    ),
    DomainConfig(
        name="Cardiology",
        dataset_name="Cardiology Medical QA",
        index_path="/kaggle/input/indexes2/Cardiology_index.faiss",
        id2doc_path="/kaggle/input/indexes2/Cardiology_docs.pkl"
    ),
    DomainConfig(
        name="Dermatology",
        dataset_name="Dermatology Medical QA",
        index_path="/kaggle/input/indexes2/dermatology_index.faiss",
        id2doc_path="/kaggle/input/indexes2/Dermatology_docs.pkl"   
    ),
    DomainConfig(
        name="Diabetes-Digestive-Kidney",
        dataset_name="Diabetes/Digestive/Kidney Medical QA",
        index_path="/kaggle/input/indexes2/Diabetes-Digestive-Kidney_index.faiss",
        id2doc_path="/kaggle/input/indexes2/Diabetes-Digestive-Kidney_docs.pkl"
    ),
    DomainConfig(
        name="Neurology",
        dataset_name="Neurology Medical QA",
        index_path="/kaggle/input/indexes2/Neurology_index.faiss",
        id2doc_path="/kaggle/input/indexes2/Neurology_docs.pkl"
    ),
]

UNIFIED_METADATA_PATH = "/kaggle/input/indexes2/metadata.json"

# =============================================================================
# RAG CONFIGURATION
# =============================================================================

class RAGConfig:
    EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
    RERANK_MODEL = "BAAI/bge-reranker-large"
    HYDE_MODEL = "google/flan-t5-large"
    GENERATOR_MODEL = "google/flan-t5-large"
    
    FAISS_TOP_K = 50
    BM25_TOP_K = 50
    FINAL_TOP_K = 8
    
    FAISS_WEIGHT = 0.6
    BM25_WEIGHT = 0.4
    QUERY_WEIGHT = 0.6
    HYDE_WEIGHT = 0.4
    
    MAX_CONTEXT_LENGTH = 512
    MAX_ANSWER_LENGTH = 256
    TEMPERATURE = 0.3
    NUM_BEAMS = 4
    DO_SAMPLE = False

config = RAGConfig()

print(f"✅ Configuration loaded")
print(f"📊 Total domains: {len(DOMAINS)}")
print(f"🤖 Models ready")


🔧 Using device: cuda
✅ Configuration loaded
📊 Total domains: 13
🤖 Models ready


In [2]:
# ======================== CELL 3: ENHANCED PIPELINE (ALL FIXES) ==========================

class MultiDomainRAGPipeline:
    """
    ✅ PRODUCTION-READY: Smart routing + Beautiful answers + No gibberish
    
    ENHANCEMENTS:
    • Keyword-based domain routing (FIX 1)
    • Multi-domain search (FIX 2)
    • Professional answer formatting (FIX 3)
    • Gibberish detection & removal
    """
    
    def __init__(self, config: RAGConfig, domains: List[DomainConfig], unified_metadata_path: str):
        self.config = config
        self.domains = {}
        self.domain_configs = {d.name: d for d in domains}
        self.unified_metadata_path = unified_metadata_path
        
        print("="*80)
        print("🏥 INITIALIZING ENHANCED MEDICAL RAG SYSTEM")
        print("="*80)
        
        self._load_unified_metadata()
        self._load_models()
        self._load_domain_indexes(domains)
        
        print(f"\n✅ Pipeline initialized with {len(self.domains)} domains")
        print("="*80)
    
    def _load_unified_metadata(self):
        """Load unified metadata.json (optional)"""
        print("\n📂 Loading unified metadata...")
        
        try:
            with open(self.unified_metadata_path, 'r') as f:
                self.unified_metadata = json.load(f)
            
            print(f"  ✅ Loaded metadata for {self.unified_metadata.get('num_domains', 0)} domains")
            
        except Exception as e:
            print(f"  ⚠️  Warning: {e}")
            print(f"  ℹ️  System will work without metadata")
            self.unified_metadata = {}
    
    def _load_models(self):
        """Load all required models"""
        print("\n📦 Loading models...")
        
        print(f"  Loading embedder: {self.config.EMBED_MODEL}")
        self.embedder = SentenceTransformer(self.config.EMBED_MODEL, device=device)
        
        print(f"  Loading reranker: {self.config.RERANK_MODEL}")
        self.reranker = CrossEncoder(self.config.RERANK_MODEL, device=device)
        
        print(f"  Loading T5-Flan: {self.config.HYDE_MODEL}")
        self.hyde_tokenizer = AutoTokenizer.from_pretrained(self.config.HYDE_MODEL)
        self.hyde_model = AutoModelForSeq2SeqLM.from_pretrained(self.config.HYDE_MODEL).to(device)
        
        self.generator_tokenizer = self.hyde_tokenizer
        self.generator_model = self.hyde_model
        
        print("  ✅ All models loaded successfully")
    
    def _load_domain_indexes(self, domains: List[DomainConfig]):
        """Load indexes with dict format support"""
        print("\n📂 Loading domain indexes...")
        
        for domain_config in domains:
            try:
                if not os.path.exists(domain_config.index_path):
                    print(f"  ⚠️  Skipping {domain_config.name} (index not found)")
                    continue
                
                if not os.path.exists(domain_config.id2doc_path):
                    print(f"  ⚠️  Skipping {domain_config.name} (pkl not found)")
                    continue
                
                print(f"  Loading {domain_config.name}...")
                
                index = faiss.read_index(domain_config.index_path)
                
                with open(domain_config.id2doc_path, 'rb') as f:
                    id2doc_raw = pickle.load(f)
                
                # Handle both string and dict formats
                id2doc = []
                if isinstance(id2doc_raw, list):
                    for item in id2doc_raw:
                        if isinstance(item, str):
                            id2doc.append(item)
                        elif isinstance(item, dict):
                            text = (item.get('text') or 
                                   item.get('content') or 
                                   item.get('answer') or 
                                   item.get('response') or 
                                   item.get('output') or
                                   str(item))
                            id2doc.append(text)
                        else:
                            id2doc.append(str(item))
                else:
                    id2doc = [str(id2doc_raw)]
                
                if not id2doc:
                    print(f"    ❌ No documents found")
                    continue
                
                domain_metadata = {}
                if 'vector_db_stats' in self.unified_metadata:
                    if domain_config.name in self.unified_metadata['vector_db_stats']:
                        domain_metadata = self.unified_metadata['vector_db_stats'][domain_config.name]
                
                tokenized_corpus = []
                for doc in id2doc:
                    try:
                        tokenized_corpus.append(word_tokenize(str(doc).lower()))
                    except:
                        tokenized_corpus.append([])
                
                bm25 = BM25Okapi(tokenized_corpus)
                
                self.domains[domain_config.name] = {
                    'config': domain_config,
                    'faiss_index': index,
                    'bm25_index': bm25,
                    'id2doc': id2doc,
                    'metadata': domain_metadata
                }
                
                print(f"    ✅ Loaded {len(id2doc)} chunks")
                
            except Exception as e:
                print(f"    ❌ Failed loading {domain_config.name}: {e}")
                continue
        
        if len(self.domains) == 0:
            raise RuntimeError("No domains loaded! Check your paths.")
    
    def route_to_domains(self, query: str) -> List[str]:
        """
        ✅ FIX 1: SMART KEYWORD-BASED ROUTING
        Routes queries to correct domains using keywords + embeddings
        """
        query_lower = query.lower()
        
        # ═══════════════════════════════════════════════════════════
        # STEP 1: Define domain keywords
        # ═══════════════════════════════════════════════════════════
        domain_keywords = {
            'drug_info': ['drug', 'medication', 'medicine', 'pill', 'prescription', 'dosage', 
                         'side effect', 'interaction', 'antibiotic', 'painkiller', 'tablet'],
            'general_medical': ['health', 'medical', 'doctor', 'hospital', 'treatment', 'symptom',
                               'disease', 'condition', 'diagnosis', 'therapy'],
            'mental_health': ['anxiety', 'panic', 'depression', 'stress', 'mental', 'mood',
                             'ptsd', 'therapy', 'counseling', 'psychology', 'emotional'],
            'ophthalmology': ['eye', 'vision', 'sight', 'blind', 'cataract', 'glaucoma',
                             'retina', 'cornea', 'glasses', 'floater'],
            'pediatrics': ['child', 'children', 'baby', 'infant', 'kid', 'toddler',
                          'pediatric', 'newborn', 'adolescent'],
            'symptoms_triage': ['fever', 'pain', 'emergency', 'urgent', 'severe', 'acute',
                               'er', 'hospital now', 'immediate'],
            'women_health': ['period', 'menstrual', 'pregnancy', 'pregnant', 'breast',
                            'ovary', 'uterus', 'menopause', 'contraceptive', 'pcos'],
            'Cancer': ['cancer', 'tumor', 'malignant', 'oncology', 'chemotherapy',
                      'radiation', 'biopsy', 'metastasis', 'carcinoma'],
            'Cardiology': ['heart', 'cardiac', 'blood pressure', 'hypertension', 'cholesterol',
                          'chest pain', 'arrhythmia', 'angina', 'stroke'],
            'Dermatology': ['skin', 'rash', 'acne', 'eczema', 'psoriasis', 'dermatitis',
                           'mole', 'lesion', 'itching', 'pimple'],
            'Diabetes-Digestive-Kidney': ['diabetes', 'sugar', 'insulin', 'kidney', 'digestive',
                                          'stomach', 'intestine', 'liver', 'pancreas'],
            'Neurology': ['brain', 'neurological', 'seizure', 'migraine', 'headache',
                         'alzheimer', 'parkinson', 'epilepsy', 'nerve']
        }
        
        # ═══════════════════════════════════════════════════════════
        # STEP 2: Count keyword matches for each domain
        # ═══════════════════════════════════════════════════════════
        keyword_scores = {}
        for domain_name in self.domains.keys():
            if domain_name in domain_keywords:
                keywords = domain_keywords[domain_name]
                matches = sum(1 for kw in keywords if kw in query_lower)
                keyword_scores[domain_name] = matches
            else:
                keyword_scores[domain_name] = 0
        
        # ═══════════════════════════════════════════════════════════
        # STEP 3: If strong keyword match (≥2), use it immediately
        # ═══════════════════════════════════════════════════════════
        max_keyword_score = max(keyword_scores.values())
        
        if max_keyword_score >= 2:
            # Strong keyword match found!
            top_domains = [name for name, score in keyword_scores.items() 
                          if score >= max(2, max_keyword_score - 1)]
            return top_domains[:2]  # Return top 2 with strong keyword match
        
        # ═══════════════════════════════════════════════════════════
        # STEP 4: Fallback to embedding-based routing
        # ═══════════════════════════════════════════════════════════
        query_emb = self.embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
        
        scores = []
        for domain_name, domain_data in self.domains.items():
            id2doc = domain_data['id2doc']
            sample_docs = id2doc[:min(50, len(id2doc))]  # Reduced from 100 for speed
            domain_embs = self.embedder.encode(sample_docs, normalize_embeddings=True, convert_to_numpy=True)
            centroid = np.mean(domain_embs, axis=0, keepdims=True)
            
            similarity = np.dot(query_emb, centroid.T)[0][0]
            scores.append((domain_name, float(similarity)))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        
        # Return top 2 domains
        selected = [name for name, score in scores[:2] if score > 0.25]
        
        if not selected:
            selected = [scores[0][0]]  # At least return top domain
        
        return selected
    
    def generate_hyde(self, query: str) -> str:
        """Generate hypothetical document"""
        try:
            prompt = f"""Generate a detailed medical answer:

Question: {query}

Answer:"""
            
            inputs = self.hyde_tokenizer(
                prompt, 
                return_tensors="pt", 
                max_length=256, 
                truncation=True
            ).to(device)
            
            with torch.no_grad():
                outputs = self.hyde_model.generate(
                    **inputs,
                    max_new_tokens=150,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.9,
                    pad_token_id=self.hyde_tokenizer.pad_token_id,
                    eos_token_id=self.hyde_tokenizer.eos_token_id
                )
            
            hyde_answer = self.hyde_tokenizer.decode(outputs[0], skip_special_tokens=True)
            return hyde_answer.strip()
            
        except Exception as e:
            return ""
    
    def hybrid_retrieval(self, query: str, hyde_text: str, domain_names: List[str]) -> List[Dict]:
        """Hybrid retrieval combining FAISS and BM25"""
        blended_query = f"{query} {hyde_text}" if hyde_text else query
        
        all_candidates = []
        
        for domain_name in domain_names:
            if domain_name not in self.domains:
                continue
            
            domain_data = self.domains[domain_name]
            faiss_index = domain_data['faiss_index']
            bm25_index = domain_data['bm25_index']
            id2doc = domain_data['id2doc']
            
            # FAISS search
            query_emb = self.embedder.encode([blended_query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
            D, I = faiss_index.search(query_emb, self.config.FAISS_TOP_K)
            
            faiss_results = {idx: float(score) for idx, score in zip(I[0], D[0]) if idx < len(id2doc)}
            
            # BM25 search
            tokenized_query = word_tokenize(blended_query.lower())
            bm25_scores = bm25_index.get_scores(tokenized_query)
            top_bm25 = np.argsort(bm25_scores)[::-1][:self.config.BM25_TOP_K]
            
            bm25_results = {int(idx): float(bm25_scores[idx]) for idx in top_bm25 if idx < len(id2doc)}
            
            # Normalize and combine
            max_faiss = max(faiss_results.values()) if faiss_results else 1.0
            max_bm25 = max(bm25_results.values()) if bm25_results else 1.0
            
            all_indices = set(faiss_results.keys()) | set(bm25_results.keys())
            
            for idx in all_indices:
                faiss_score = faiss_results.get(idx, 0.0) / max_faiss
                bm25_score = bm25_results.get(idx, 0.0) / max_bm25
                
                combined_score = (
                    self.config.FAISS_WEIGHT * faiss_score +
                    self.config.BM25_WEIGHT * bm25_score
                )
                
                all_candidates.append({
                    'domain': domain_name,
                    'chunk': id2doc[idx],
                    'score': combined_score
                })
        
        all_candidates.sort(key=lambda x: x['score'], reverse=True)
        return all_candidates[:30]
    
    def rerank_results(self, query: str, candidates: List[Dict]) -> List[Dict]:
        """Rerank using cross-encoder"""
        if not candidates:
            return []
        
        pairs = [[query, c['chunk']] for c in candidates]
        rerank_scores = self.reranker.predict(pairs)
        
        for i, cand in enumerate(candidates):
            cand['rerank_score'] = float(rerank_scores[i])
        
        candidates.sort(key=lambda x: x['rerank_score'], reverse=True)
        return candidates[:self.config.FINAL_TOP_K]
    
    def _clean_gibberish(self, text: str) -> str:
        """
        ✅ FIX 3: Remove gibberish patterns
        """
        # Remove common gibberish patterns
        gibberish_patterns = [
            r'Chat Doctor',
            r'I am Chat Doctor',
            r'with Chat Doctor',
            r'Alma\b',
            r'hyper Alma',
            r'Chat\s+Doctor',
            r'\bchat\s+doctor\b',
        ]
        
        cleaned = text
        for pattern in gibberish_patterns:
            cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE)
        
        # Remove multiple spaces
        cleaned = re.sub(r'\s+', ' ', cleaned)
        
        return cleaned.strip()
    
    def _format_professional_answer(self, answer: str) -> str:
        """
        ✅ FIX 3: Format answer like your friend's output
        Beautiful paragraphs + bullet points + professional structure
        """
        # Clean gibberish first
        answer = self._clean_gibberish(answer)
        
        # Split into sentences
        sentences = sent_tokenize(answer)
        
        # Group into paragraphs (every 2-3 sentences)
        paragraphs = []
        current_para = []
        
        for sent in sentences:
            if len(sent) > 15:  # Valid sentence
                current_para.append(sent)
                if len(current_para) >= 2:  # Make paragraph
                    paragraphs.append(' '.join(current_para))
                    current_para = []
        
        if current_para:
            paragraphs.append(' '.join(current_para))
        
        # Format with proper line breaks
        formatted = '\n\n'.join(paragraphs)
        
        # Add disclaimer
        if len(formatted) > 50:
            formatted += "\n\n⚠️ Important: This information is for educational purposes only. Please consult a healthcare professional for personalized medical advice."
        
        return formatted
    
    def _extractive_fallback(self, context_chunks: List[Dict]) -> str:
        """Professional extractive fallback"""
        if not context_chunks:
            return (
                "I apologize, but I couldn't find specific medical information to answer your question.\n\n"
                "⚠️ Please consult a healthcare professional for accurate medical advice."
            )
        
        best_chunk = context_chunks[0]['chunk'].strip()
        best_chunk = self._clean_gibberish(best_chunk)
        
        sentences = sent_tokenize(best_chunk)
        complete_sentences = [
            s for s in sentences 
            if len(s) > 15 and s.strip()[-1] in '.!?'
        ]
        
        if complete_sentences:
            answer = ' '.join(complete_sentences[:4])  # First 4 sentences
        else:
            answer = best_chunk[:300]  # First 300 chars
        
        answer += "\n\n⚠️ Important: Please consult a healthcare professional for personalized medical advice."
        
        return answer
    
    def generate_answer(self, query: str, context_chunks: List[Dict]) -> str:
        """
        ✅ FIX 3: Generate BEAUTIFUL formatted answers
        """
        if not context_chunks:
            return self._extractive_fallback([])
        
        # Build context from high-quality chunks only
        context_parts = []
        for i, chunk_data in enumerate(context_chunks[:3], 1):
            if chunk_data['rerank_score'] > 0.80:  # Increased threshold
                chunk_text = chunk_data['chunk'].strip()
                chunk_text = self._clean_gibberish(chunk_text)
                context_parts.append(f"[Source {i}]: {chunk_text}")
        
        if not context_parts:
            return self._extractive_fallback(context_chunks)
        
        combined_context = "\n\n".join(context_parts)
        
        if len(combined_context) > 1800:
            combined_context = combined_context[:1800] + "..."
        
        # Enhanced prompt for better answers
        prompt = f"""Answer the medical question professionally based on the provided context. 
Format your answer with clear paragraphs. Be concise but comprehensive.

Context:
{combined_context}

Question: {query}

Professional Answer:"""
        
        try:
            inputs = self.generator_tokenizer(
                prompt,
                return_tensors="pt",
                max_length=self.config.MAX_CONTEXT_LENGTH,
                truncation=True
            ).to(device)
            
            with torch.no_grad():
                outputs = self.generator_model.generate(
                    **inputs,
                    max_new_tokens=self.config.MAX_ANSWER_LENGTH,
                    temperature=0.3,  # Lower for more factual
                    num_beams=4,
                    do_sample=False,
                    early_stopping=True,
                    pad_token_id=self.generator_tokenizer.pad_token_id,
                    eos_token_id=self.generator_tokenizer.eos_token_id
                )
            
            answer = self.generator_tokenizer.decode(outputs[0], skip_special_tokens=True)
            answer = answer.strip()
            
            # Post-process
            if "Answer:" in answer or "Professional Answer:" in answer:
                answer = answer.split(":")[-1].strip()
            
            # Clean and format
            answer = self._format_professional_answer(answer)
            
            # Validation
            if (len(answer) < 40 or 
                "i don't know" in answer.lower() or
                "no information" in answer.lower() or
                "cannot answer" in answer.lower()):
                return self._extractive_fallback(context_chunks)
            
            return answer
        
        except Exception as e:
            print(f"⚠️  Generation failed: {e}")
            return self._extractive_fallback(context_chunks)
    
    def compute_metrics(self, query: str, answer: str, context_chunks: List[Dict]) -> Dict:
        """Compute confidence metrics"""
        retrieval_score = np.mean([c['rerank_score'] for c in context_chunks]) if context_chunks else 0.0
        
        answer_emb = self.embedder.encode([answer], normalize_embeddings=True, convert_to_numpy=True)
        context_text = " ".join([c['chunk'] for c in context_chunks])
        context_emb = self.embedder.encode([context_text], normalize_embeddings=True, convert_to_numpy=True)
        faithfulness = float(np.dot(answer_emb, context_emb.T)[0][0])
        
        composite = 0.6 * retrieval_score + 0.4 * faithfulness
        
        return {
            'retrieval_score': float(retrieval_score),
            'faithfulness': float(faithfulness),
            'composite': float(composite)
        }
    
    def run_query(self, query: str) -> Dict:
        """Main query pipeline"""
        start_time = time.time()
        
        print(f"\n🔍 Query: {query}")
        
        selected_domains = self.route_to_domains(query)
        print(f"📍 Domains: {', '.join(selected_domains)}")
        
        print("🔮 Generating HyDE...")
        hyde_text = self.generate_hyde(query)
        
        print("🔎 Hybrid retrieval...")
        candidates = self.hybrid_retrieval(query, hyde_text, selected_domains)
        print(f"   Retrieved {len(candidates)} candidates")
        
        if not candidates:
            return {
                'query': query,
                'answer': self._extractive_fallback([]),
                'domains': selected_domains,
                'sources': [],
                'metrics': {'composite': 0.0},
                'processing_time': time.time() - start_time
            }
        
        print("🎯 Reranking...")
        top_chunks = self.rerank_results(query, candidates)
        
        print("💬 Generating professional answer...")
        answer = self.generate_answer(query, top_chunks)
        
        metrics = self.compute_metrics(query, answer, top_chunks)
        
        processing_time = time.time() - start_time
        print(f"✅ Done in {processing_time:.2f}s (confidence: {metrics['composite']:.2f})")
        
        return {
            'query': query,
            'answer': answer,
            'domains': selected_domains,
            'sources': [{'chunk': c['chunk'][:200], 'domain': c['domain'], 'score': c['rerank_score']} 
                       for c in top_chunks],
            'metrics': metrics,
            'processing_time': processing_time
        }

print("✅ Enhanced MultiDomainRAGPipeline: Smart routing + Beautiful answers + No gibberish")


✅ Enhanced MultiDomainRAGPipeline: Smart routing + Beautiful answers + No gibberish


In [3]:
# ======================== CELL 4: INITIALIZE PIPELINE ==========================

print("\n" + "="*80)
print("🚀 INITIALIZING PIPELINE")
print("="*80 + "\n")

# ✅ CORRECTED: Pass unified_metadata_path
pipeline = MultiDomainRAGPipeline(config, DOMAINS, UNIFIED_METADATA_PATH)

print("\n" + "="*80)
print("✅ PIPELINE READY WITH T5-FLAN!")
print("="*80)



🚀 INITIALIZING PIPELINE

🏥 INITIALIZING ENHANCED MEDICAL RAG SYSTEM

📂 Loading unified metadata...
  ✅ Loaded metadata for 5 domains

📦 Loading models...
  Loading embedder: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

  ✅ All models loaded successfully

📂 Loading domain indexes...
  Loading drug_info...
    ✅ Loaded 435395 chunks
  Loading general_medical...
    ✅ Loaded 710919 chunks
  Loading mental_health...
    ✅ Loaded 22565 chunks
  Loading ophthalmology...
    ✅ Loaded 57979 chunks
  Loading pediatrics...
    ✅ Loaded 19888 chunks
  Loading medical_qa...
    ✅ Loaded 777049 chunks
  Loading symptoms_triage...
    ✅ Loaded 147907 chunks
  Loading women_health...
    ✅ Loaded 236304 chunks
  Loading Cancer...
    ✅ Loaded 729 chunks
  Loading Cardiology...
    ✅ Loaded 5000 chunks
  Loading Dermatology...
    ✅ Loaded 1460 chunks
  Loading Diabetes-Digestive-Kidney...
    ✅ Loaded 1192 chunks
  Loading Neurology...
    ✅ Loaded 1452 chunks

✅ Pipeline initialized with 13 domains

✅ PIPELINE READY WITH T5-FLAN!


In [4]:
# ======================== CELL 5: INTERACTIVE MODE ==========================

def ask_question():
    """Interactive mode - ask questions one by one"""
    print("\n" + "="*80)
    print("💬 INTERACTIVE MEDICAL QA MODE")
    print("="*80)
    print("Type your medical questions below.")
    print("Type 'quit' or 'exit' to stop.\n")
    
    while True:
        # Get user input
        query = input("\n🔍 Your Question: ").strip()
        
        if not query:
            print("⚠️  Please enter a question")
            continue
        
        if query.lower() in ['quit', 'exit', 'stop', 'q']:
            print("\n👋 Goodbye!")
            break
        
        print("\n" + "-"*80)
        
        try:
            # Process query
            result = pipeline.run_query(query)
            
            # Display answer
            print(f"\n💡 **ANSWER:**")
            print(f"{result['answer']}\n")
            
            # Display metadata
            print(f"📊 Confidence: {result['metrics']['composite']:.2f}")
            print(f"🎯 Knowledge Domains: {', '.join(result['domains'])}")
            print(f"⏱️  Response Time: {result['processing_time']:.2f}s")
            
            # Show sources
            if result['sources']:
                show_sources = input("\n📚 Show sources? (y/n): ").strip().lower()
                if show_sources == 'y':
                    print("\nTop Sources:")
                    for i, source in enumerate(result['sources'][:3], 1):
                        print(f"\n{i}. [{source['domain']}] Relevance: {source['score']:.2f}")
                        print(f"   {source['chunk']}")
        
        except Exception as e:
            print(f"\n❌ Error processing query: {e}")
            print("Please try again with a different question.")
        
        print("\n" + "-"*80)

# Run interactive mode
ask_question()



💬 INTERACTIVE MEDICAL QA MODE
Type your medical questions below.
Type 'quit' or 'exit' to stop.




🔍 Your Question:  I'm a 32-year-old woman with severe anxiety. I'm also on metformin for diabetes  and just noticed a strange rash on my hands. What could be causing this and should I  be concerned about drug interactions?



--------------------------------------------------------------------------------

🔍 Query: I'm a 32-year-old woman with severe anxiety. I'm also on metformin for diabetes  and just noticed a strange rash on my hands. What could be causing this and should I  be concerned about drug interactions?
📍 Domains: drug_info, symptoms_triage
🔮 Generating HyDE...
🔎 Hybrid retrieval...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

✅ Done in 5.04s (confidence: 0.50)

💡 **ANSWER:**
She has a fever of 104°F, stiff neck, purple spots on her legs, and is sleeping more than usual.

⚠️ Important: This information is for educational purposes only. Please consult a healthcare professional for personalized medical advice.

📊 Confidence: 0.50
🎯 Knowledge Domains: symptoms_triage
⏱️  Response Time: 5.04s



📚 Show sources? (y/n):  y



Top Sources:

1. [symptoms_triage] Relevance: 0.93
   The fever lasts longer than 5 days. It remains high even after treatment with standard childhood fever medicines. Other classic signs of the disease are:
                
Swollen lymph nodes in the ne

2. [symptoms_triage] Relevance: 0.53
   It remains high even after treatment with standard childhood fever medicines. Other classic signs of the disease are:
                
Swollen lymph nodes in the neck
                
A rash on the mi

3. [symptoms_triage] Relevance: 0.49
   The fever remains high even after treatment with standard childhood fever medicines. Children who have the disease also may have red eyes, red lips, and redness on the palms of their hands and soles o

--------------------------------------------------------------------------------



🔍 Your Question:  exit



👋 Goodbye!


In [ ]:
# ======================== CELL 6: SAVE RESULTS & EXPORT ==========================

# Save sample results to JSON
sample_results = []

test_queries = [
    "What is diabetes?",
    "How to manage anxiety?",
    "Child fever treatment"
]

for query in test_queries:
    result = pipeline.run_query(query)
    sample_results.append({
        'question': query,
        'answer': result['answer'],
        'confidence': result['metrics']['composite'],
        'domains': result['domains']
    })

# Save to file
with open('sample_results.json', 'w') as f:
    json.dump(sample_results, f, indent=2)

print("✅ Sample results saved to sample_results.json")
print("\n📦 TO EXPORT THIS NOTEBOOK:")
print("1. Click the three dots (...) in top right")
print("2. Select 'Download notebook as .py'")
print("3. Send the .py file + all index files to Nikhil")
